# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)


> **Copia local de la imagen** (por si el enlace de Drive no carga): `img/warehouse.png`
>
> ![Mundo del ejercicio (copia local)](img/warehouse.png)
>
> **Lectura de la imagen** (coordenadas `(fila, columna)`, origen arriba a la izquierda):
>
> | Elemento | Celdas |
> |---|---|
> | `START` | (0,0) |
> | Estanterías / paredes | (0,3), (1,1), (2,4), (4,2) |
> | Entrega `+10` (terminal) | (0,5) |
> | Carga `+2` (terminal) | (2,2) |
> | Peligro mortal `-10` (terminal) | (3,5) |
> | Peligro `-3` (no terminal) | (1,4), (4,1) |
> | Piso resbaloso | (1,2), (2,1), (3,3) |


## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, slippery_intended=0.60):
        self.height = 5
        self.width = 6

        # Leído de la imagen del ejercicio (fila, columna), origen arriba-izquierda.
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}          # estanterías
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}         # piso resbaloso

        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (2, 2): +2.0,    # estación de carga
            (3, 5): -10.0,   # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,    # peligro no terminal
            (4, 1): -3.0,    # peligro no terminal
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # Dinámica: P(dirección elegida) y P(desviación) a cada lado.
        self.normal_probs = (0.90, 0.05)
        self.slippery_probs = (slippery_intended, (1.0 - slippery_intended) / 2)

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa del estado actual.
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Estado terminal: absorbente.
        if self.is_terminal(state):
            return [(state, 1.0)]

        p_intended, p_side = (
            self.slippery_probs if state in self.slippery_states
            else self.normal_probs
        )

        # Desviaciones a la izquierda y a la derecha de la acción elegida.
        dr, dc = action
        left_of = (-dc, dr)
        right_of = (dc, -dr)

        outcomes = [
            (action, p_intended),
            (left_of, p_side),
            (right_of, p_side),
        ]

        transitions = []
        for move, prob in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])
            # Borde o estantería: el robot permanece en el mismo estado.
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # V_0(s) = 0 para todos los estados.
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()            # actualización sincrónica
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # V_{k+1}(s) = R(s) + gamma max_a sum T(s,a,s') V_k(s')
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = grid.get_reward(state) + grid.gamma * best_expected_value

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # V^pi_{k+1}(s) = R(s) + gamma sum T(s,pi(s),s') V^pi_k(s')   (sin max)
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum T(s,a,s') V^pi(s')
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria: RIGHT en todos los estados no terminales
    policy = {
        state: (0, 1)
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for iteration in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(new_policy[s] != policy[s] for s in new_policy)
        history.append(changed)
        policy = new_policy

        # 4. repetir hasta estabilidad
        if changed == 0:
            break

    V = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [19, 7, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Respuestas — Parte 5 (con la configuración base: `living_reward=-1`, `gamma=0.9`, resbaloso 0.60)

Los valores citados salen de la ejecución de las celdas anteriores.

**1. Desde `START`, ¿busca la entrega +10 o la carga +2?**
Busca la **estación de carga +2**. La política desde `START` es → → ↓ ↓: (0,0)→(0,1)→(0,2)→(1,2)→(2,2). El valor del inicio es $V(\text{START})\approx-2.57$.

**2. ¿Por qué una recompensa menor puede ser óptima?**
El agente maximiza el **retorno descontado esperado**, no la recompensa final. Ir a +10 requiere unos 7 pasos que cuestan $-1$ cada uno, pasar junto a peligros (el camino corto cruza el $-3$ en (1,4)) y el $+10$ llega descontado ($0.9^7\approx0.48$). Llegar a +2 cuesta unos 4 pasos. Con $\gamma=0.9$ y costo $-1$, el retorno esperado de la ruta a +2 es mayor que el de la ruta a +10.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
Comparando con el mismo MDP sin piso resbaloso (verificación en la celda de abajo), la política cambia en:
- **(1,2)**, que es resbaloso: con piso normal elegiría → (hacia la entrega); con piso resbaloso elige ↓ y entra directamente a la carga. Una desviación del 20 % hacia (1,3) o (1,1) no es grave, pero seguir hacia la entrega desde un piso inestable vale menos que asegurar el +2.
- **(3,1)**, que no es resbaloso pero es vecino de (2,1): con piso normal subiría ↑ pasando por (2,1); como (2,1) es resbaloso (desviación del 40 %), esa ruta vale menos y la política prefiere →.

**4. ¿Qué papel cumple el costo por paso `-1`?**
Penaliza cada paso no terminal: induce rutas cortas y hace costoso "dar la vuelta" hasta la recompensa lejana. Con un costo tan alto, la meta cercana (+2) gana. En el Experimento A se ve que, al bajarlo a $-0.1$, la decisión cambia.

**5. ¿Por qué $T(s,a,s')$ ya no puede usar las mismas probabilidades en todos los estados?**
Porque la dinámica depende del **tipo de piso del estado actual**: en piso normal $P(\text{elegida})=0.9$, en resbaloso $0.6$. $T$ debe consultar `state in slippery_states` antes de repartir la probabilidad; una sola tabla 0.8/0.1/0.1 como en GridWorld ya no describe el mundo.

In [7]:
# Verificación para la pregunta 3: mismo MDP sin piso resbaloso
grid_no_slip = WarehouseMDP()
grid_no_slip.slippery_states = set()
V_ns, _ = value_iteration(grid_no_slip)
pi_ns = extract_policy(grid_no_slip, V_ns)

print("Estados donde cambia la acción (resbaloso -> sin resbaloso):")
for s in pi_vi:
    if pi_vi[s] != pi_ns[s]:
        print(f"  {s}: {ARROWS[pi_vi[s]]}  ->  {ARROWS[pi_ns[s]]}")

Estados donde cambia la acción (resbaloso -> sin resbaloso):
  (1, 2): ↓  ->  →
  (3, 1): →  ->  ↑


In [8]:
def nominal_route(grid, policy, max_steps=30):
    """Ruta si cada acción tuviera el resultado deseado (solo para describir la política)."""
    s = grid.start
    route = [s]
    while not grid.is_terminal(s) and len(route) < max_steps:
        a = policy[s]
        nxt = (s[0] + a[0], s[1] + a[1])
        s = nxt if grid.is_valid_state(nxt) else s
        route.append(s)
    return route


def run_experiment(title, **params):
    g = WarehouseMDP(**params)
    V, n = value_iteration(g)
    pi = extract_policy(g, V)
    pi_check, _, _ = policy_iteration(g)
    print(f"=== {title} {params} ===")
    print(f"Iteraciones VI: {n} | VI == PI: {pi == pi_check}")
    print(f"V(START) = {V[g.start]:+.3f}")
    print_policy(g, pi)
    route = nominal_route(g, pi)
    print("Ruta nominal desde START:", route, "-> termina en", g.get_reward(route[-1]))
    return g, V, pi

_ = run_experiment("BASE")

=== BASE {} ===
Iteraciones VI: 20 | VI == PI: True
V(START) = -2.575
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Ruta nominal desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)] -> termina en 2.0


### Experimento A — ejecución

**Predicción (antes de ejecutar):** con un costo por paso 10 veces menor, caminar más ya casi no penaliza; esperamos que desde `START` el robot prefiera la entrega +10.

In [9]:
_ = run_experiment("Experimento A", living_reward=-0.1)

=== Experimento A {'living_reward': -0.1} ===
Iteraciones VI: 26 | VI == PI: True
V(START) = +1.649
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Ruta nominal desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)] -> termina en 10.0


**Resultado:** se confirma. Desde `START` la ruta pasa a ser (0,0)→(0,1)→(0,2)→(1,2)→(1,3)→(1,4)→(1,5)→(0,5): el robot va a la **entrega +10** y $V(\text{START})$ sube de $-2.57$ a $+1.65$. También cambian (1,0), (1,2) y (3,2), que ahora se orientan hacia la entrega. Llama la atención que la ruta cruza el peligro $-3$ de (1,4): rodearlo por la fila 0 no es posible porque (0,3) es estantería.

### Experimento B — ejecución

**Predicción:** el piso resbaloso se vuelve más riesgoso (40 % de acertar); la política base ya lo evitaba casi siempre, así que esperamos que la política no cambie y solo bajen los valores cercanos a las celdas resbalosas.

In [10]:
_ = run_experiment("Experimento B", slippery_intended=0.40)

=== Experimento B {'slippery_intended': 0.4} ===
Iteraciones VI: 22 | VI == PI: True
V(START) = -2.706
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Ruta nominal desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)] -> termina en 2.0


**Resultado:** la política es **idéntica** a la base; solo bajan los valores ($V(\text{START})$ pasa de $-2.57$ a $-2.71$). Con $-1$ por paso la meta sigue siendo la carga +2, y desde (1,2) moverse ↓ sigue siendo la mejor apuesta aunque solo acierte el 40 %, porque las desviaciones (a (1,1) pared → se queda, o a (1,3)) no llevan a terminales negativos.

### Experimento C — ejecución

**Predicción:** con $\gamma=0.99$ el +10 lejano casi no se descuenta ($0.99^7\approx0.93$); esperamos que el agente valore más la entrega.

In [11]:
_ = run_experiment("Experimento C", gamma=0.99)

=== Experimento C {'gamma': 0.99} ===
Iteraciones VI: 24 | VI == PI: True
V(START) = -1.456
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Ruta nominal desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)] -> termina en 10.0


**Resultado:** sí. Con $\gamma=0.99$ la ruta desde `START` termina en la **entrega +10** y $V(\text{START})$ sube a $-1.46$. La política cambia en (1,2), que ahora va → hacia la entrega en vez de ↓ a la carga. Más paciencia (mayor $\gamma$) hace que la recompensa lejana pese más que la cercana.

### Bonus — umbral de `living_reward`

Barremos `living_reward` de $-2.00$ a $0.00$ en pasos de $0.01$ y registramos en qué terminal acaba la ruta nominal desde `START`.

In [12]:
import numpy as np

previous = None
for lr in np.round(np.arange(-2.00, 0.001, 0.01), 2):
    g = WarehouseMDP(living_reward=float(lr))
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    end = nominal_route(g, pi)[-1]
    if end != previous:
        print(f"living_reward = {lr:+.2f} -> termina en {end} (R = {g.get_reward(end):+.0f})")
        previous = end

living_reward = -2.00 -> termina en (2, 2) (R = +2)


living_reward = -0.79 -> termina en (0, 5) (R = +10)


**Resultado:** para `living_reward` $\le -0.80$ el robot va a la carga +2; desde aproximadamente **$-0.79$** (entre $-0.80$ y $-0.79$, con $\gamma=0.9$ y resbaloso 0.60) cambia a la entrega +10. El umbral es aproximado: depende de la resolución del barrido y del criterio de parada $\theta=10^{-4}$.